# Maintenance prédictive par l'audio - entraînement (Kaggle)

Panneau *Settings* : **Accelerator = GPU T4 x2**, **Internet = On**. Puis **Save Version > Save & Run All (Commit)** : le calcul continue en arrière-plan.

Pour reprendre une exécution incomplète : *Add Input* > sortie de ce notebook, puis relancer.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'Choisir GPU T4 x2'
assert hasattr(torch.nn.utils, 'get_total_norm'), 'PyTorch >= 2.6 requis'
torch.ones(2, device='cuda').sum().item()  # echoue si le GPU n'est pas supporte
print(torch.cuda.get_device_name(), torch.__version__)
OUT, DATA = '/kaggle/working', '/tmp/mimii'
os.environ.update(MIMII_ROOT=DATA, MIMII_MODELS=f'{OUT}/models', MIMII_RESULTS=f'{OUT}/results',
                  CUDA_VISIBLE_DEVICES='0')
os.makedirs(f'{DATA}/dcase', exist_ok=True)

In [ ]:
import base64, glob, io, shutil, sys, subprocess, zipfile
CODE = '...'
shutil.rmtree('/tmp/project', ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall('/tmp/project')
os.chdir('/tmp/project')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'librosa', 'soundfile', 'scikit-learn', 'pytest'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], check=True)
# reprise : recopier les resultats d'une version precedente ajoutee en entree
for z in glob.glob('/kaggle/input/**/training_results.zip', recursive=True):
    zipfile.ZipFile(z).extractall(OUT); print('reprise depuis', z)

## Données (Zenodo, sommes MD5 vérifiées)

In [ ]:
import hashlib

def get(url, dest, md5):
    subprocess.run(['curl', '-fL', '--retry', '10', '--retry-all-errors', '-sS', '-o', dest, url], check=True)
    h = hashlib.md5()
    with open(dest, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''): h.update(b)
    assert h.hexdigest() == md5, f'MD5 incorrect : {dest}'
    print('ok', dest, flush=True)

Z = 'https://zenodo.org/records'
os.makedirs(f'{DATA}/panns', exist_ok=True)
get(f'{Z}/3987831/files/Cnn14_16k_mAP%3D0.438.pth?download=1', f'{DATA}/panns/Cnn14_16k.pth',
    '362fc5ff18f1d6ad2f6d464b45893f2c')
for m, h in (('pump', '90e7091ef722b7238a7f1009365779cd'), ('fan', '649bdfc06263ae7a838963f43b6641e6')):
    zp = f'/tmp/dev_data_{m}.zip'
    get(f'{Z}/3678171/files/dev_data_{m}.zip?download=1', zp, h)
    zipfile.ZipFile(zp).extractall(f'{DATA}/dcase'); os.remove(zp)
counts = {(m, s): len(os.listdir(f'{DATA}/dcase/{m}/{s}')) for m in ('pump', 'fan') for s in ('train', 'test')}
print(counts)
assert counts == {('pump', 'train'): 3349, ('pump', 'test'): 856, ('fan', 'train'): 3675, ('fan', 'test'): 1875}

## Entraînements

Une erreur n'arrête pas le notebook, pour que les résultats obtenus soient toujours sauvegardés.

In [ ]:
def save():
    with zipfile.ZipFile(f'{OUT}/training_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for d in ('results', 'models'):
            for p in glob.glob(f'{OUT}/{d}/**/*', recursive=True):
                if os.path.isfile(p): z.write(p, os.path.relpath(p, OUT))

def sh(*a):
    p = subprocess.Popen([sys.executable, '-u', *a], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='', flush=True)
    return p.wait()

print('code retour', sh('scripts/run_all.py', '--datasets', 'dcase')); save()

## Étude du bruit (sous-ensemble MIMII -6, 0, +6 dB)

In [ ]:
todo = [1 for d in ('mimii_-6dB', 'mimii_0dB', 'mimii_6dB') for m in ('pump', 'fan')
        for k in ('panns', 'dense', 'conv') if not os.path.exists(f'{OUT}/results/scores/{d}_{m}_{k}.csv')]
rc = sh('scripts/download_mimii_subset.py') if todo else -1  # rien a faire : pas de telechargement
n = len(glob.glob(f'{DATA}/mimii_*dB/*/id_*/*/*.wav'))
print('clips MIMII :', n, '/ 6000')
if rc == 0 and n == 6000:
    print('code retour', sh('scripts/run_all.py'))
else:
    print('MIMII deja termine' if not todo else 'MIMII incomplet : etude du bruit non lancee')
save()

In [ ]:
done = {os.path.basename(p)[:-4] for p in glob.glob(f'{OUT}/results/scores/*.csv')}
want = {f'{d}_{m}_{k}' for d in ('dcase', 'mimii_-6dB', 'mimii_0dB', 'mimii_6dB')
        for m in ('pump', 'fan') for k in ('panns', 'dense', 'conv')}
print(f'{len(done & want)}/24 termines ; manquants :', sorted(want - done))
print('Telecharger training_results.zip dans l onglet Output')